# Домашнее задание по теме «Spark SQL»

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, split, count, desc, lower

spark = SparkSession.builder.appName("homework").getOrCreate()

df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load("covid-data.csv"))

In [2]:
df.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: double (nullable = true)
 |-- new_cases: double (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: double (nullable = true)
 |-- new_deaths: double (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: double (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: double (nullable = true)
 |-- hosp_patients_per_million: 

## Задание 1. 
15 стран с наибольшим процентом переболевших на 31 марта 

In [11]:
from pyspark.sql import functions as F

result = df \
    .where(F.col('date') == '2021-03-31')\
    .filter(F.col("population").isNotNull()) \
    .withColumn("percent_infected", (F.col("total_cases") / F.col("population")) * 100) \
    .withColumn("percent_infected", F.round(F.col("percent_infected"), 2)) \
    .orderBy(F.col("percent_infected").desc()) \
    .limit(15) \
    .select("iso_code", "location", "percent_infected") 

result.show(truncate=False)

+--------+-------------+----------------+
|iso_code|location     |percent_infected|
+--------+-------------+----------------+
|AND     |Andorra      |15.54           |
|MNE     |Montenegro   |14.52           |
|CZE     |Czechia      |14.31           |
|SMR     |San Marino   |13.94           |
|SVN     |Slovenia     |10.37           |
|LUX     |Luxembourg   |9.85            |
|ISR     |Israel       |9.63            |
|USA     |United States|9.2             |
|SRB     |Serbia       |8.83            |
|BHR     |Bahrain      |8.49            |
|PAN     |Panama       |8.23            |
|PRT     |Portugal     |8.06            |
|EST     |Estonia      |8.02            |
|SWE     |Sweden       |7.97            |
|LTU     |Lithuania    |7.94            |
+--------+-------------+----------------+



## Задание 2. 
Top 10 стран с максимальным зафиксированным кол-вом новых случаев за последнюю неделю марта 2021 в отсортированном порядке по убыванию

In [17]:
task2_df = df.filter((F.col("date") >= "2021-03-25") & (F.col("date") <= "2021-03-31")) \
    .filter(~df["location"].isin("World", "Europe", "Asia", "European Union", "South America", "North America")) \
    .orderBy(F.col("сегодня").desc()) \
    .limit(10) \
    .select(
        "date", 
        "location", 
        F.col("сегодня").cast("int").alias("сегодня")
    )

task2_df.show(truncate=False)

+----------+-------------+---------+
|date      |location     |new_cases|
+----------+-------------+---------+
|2021-03-25|Brazil       |100158   |
|2021-03-31|Brazil       |90638    |
|2021-03-27|Brazil       |85948    |
|2021-03-30|Brazil       |84494    |
|2021-03-26|Brazil       |84245    |
|2021-03-26|United States|77321    |
|2021-03-31|India        |72330    |
|2021-03-29|United States|69429    |
|2021-03-28|India        |68020    |
|2021-03-25|United States|67465    |
+----------+-------------+---------+



## Задание 3. 
Посчитайте изменение случаев относительно предыдущего дня в России за последнюю неделю марта 2021. 
(например: в России вчера было 9150 , сегодня 8763, итог: -387) 
(в выходящем датасете необходимы колонки: число, кол-во новых случаев вчера, кол-во новых случаев сегодня, дельта)

In [31]:
from pyspark.sql import Window

df_russia = df.filter((F.col("location") == "Russia") & 
                      (F.col("date") >= "2021-03-25") & 
                      (F.col("date") <= "2021-03-31"))

task3_df = df_russia \
    .withColumn("случаи вчера", F.lag("new_cases").over(Window.orderBy("date"))) \
    .withColumn("дельта", F.col("new_cases") - F.col("случаи вчера")) \
    .select(
        F.col("date").alias("Дата"), 
        F.col("случаи вчера").cast("int"), 
        F.col("new_cases").cast("int").alias("сегодня"), 
        F.col("дельта").cast("int")
    )

task3_df.show(truncate=False)

+----------+------------+-------+------+
|Дата      |случаи вчера|сегодня|дельта|
+----------+------------+-------+------+
|2021-03-25|NULL        |9128   |NULL  |
|2021-03-26|9128        |9073   |-55   |
|2021-03-27|9073        |8783   |-290  |
|2021-03-28|8783        |8979   |196   |
|2021-03-29|8979        |8589   |-390  |
|2021-03-30|8589        |8162   |-427  |
|2021-03-31|8162        |8156   |-6    |
+----------+------------+-------+------+

